# Preprocesamiento de texto

**CC3084 · Ciencia de Datos · Universidad del Valle de Guatemala · Semestre II, 2026**

Este cuaderno cubre el ejercicio 3 del enunciado: la limpieza y el preprocesamiento del
texto de los tweets, documentado paso a paso con ejemplos antes y después.

El objetivo final del laboratorio es clasificar si un tweet describe un desastre real. Ese
objetivo es una tarea de tema, y una tarea de tema pide un vocabulario reducido a lo
temáticamente relevante: fuera URLs, menciones, símbolos y palabras vacías. Pero el
laboratorio también contempla un análisis de sentimiento en la entrega final, y esa es una
tarea de opinión que necesita justo lo contrario: mayúsculas, signos de puntuación
expresivos y, sobre todo, negaciones intactas. Aplicar una sola limpieza a las dos tareas
rompería una de las dos, así que este cuaderno construye y documenta **dos limpiezas
distintas**, implementadas en `src/preprocesamiento.py`:

- `limpiar_clasificacion`: agresiva, para el ejercicio 6.
- `limpiar_sentimiento`: conservadora, para los ejercicios 8 a 10 de la entrega final. Hoy
  se deja lista y probada, no se explota todavía.

Todo el código de limpieza vive en el módulo, no en este cuaderno: el cuaderno lo importa,
lo demuestra y lo aplica.

In [1]:
import sys
import math
import collections
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))

import config
from carga import cargar_tweets
from preprocesamiento import (
    limpiar_clasificacion, limpiar_sentimiento, tokenizar, pipeline_texto,
    NEGACIONES, STOPWORDS_INGLES, STOPWORDS_DOMINIO, STOPWORDS_CLASIFICACION,
    STEMMER, LEMATIZADOR,
)

config.asegurar_directorios()

pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 110)

def guardar_tabla(df, nombre):
    ruta = config.TABLES / nombre
    df.to_csv(ruta, index=False, encoding='utf-8')
    print(f'tabla guardada: {ruta.name}')
    return ruta

print('Entorno listo.')

Entorno listo.


In [2]:
df = cargar_tweets(deduplicar=True, verbose=True)
print()
print(f'Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas')

Leído train.csv: 7,613 filas x 5 columnas
Deduplicación
  Filas originales................. 7,613
  Textos con etiqueta contradictoria 18 (55 filas eliminadas)
  Duplicados exactos eliminados.... 73
  Filas finales.................... 7,485 (128 eliminadas en total)

Dimensiones: 7,485 filas x 7 columnas


## Limpieza paso a paso, con ejemplos reales del corpus

Cada paso se muestra por separado, sobre tweets reales del conjunto, para que quede claro
qué hace y por qué. El orden importa: por ejemplo el mojibake se arregla antes de buscar
entidades HTML, porque el mojibake puede esconder una entidad a medio corromper.

In [3]:
import re
from preprocesamiento import (
    RE_MOJIBAKE, RE_URL, RE_MENCION, RE_HASHTAG, RE_ENTIDAD_HTML,
    _arreglar_entidades_y_mojibake, _quitar_emoticones,
)

ejemplos_pasos = [
    ('Mojibake y entidades HTML',
     'Barbados #Bridgetown JAMAICA \x89\x9b\x92 Two cars set ablaze: SANTA CRUZ',
     _arreglar_entidades_y_mojibake('Barbados #Bridgetown JAMAICA \x89\x9b\x92 Two cars set ablaze: SANTA CRUZ')),
    ('Entidad HTML',
     'Rene Ablaze &amp; Jacinta - Secret 2k13 (Fallen Skies Edit)',
     _arreglar_entidades_y_mojibake('Rene Ablaze &amp; Jacinta - Secret 2k13 (Fallen Skies Edit)')),
    ('URL a token uniforme',
     '@bbcmtd Wholesale Markets ablaze http://t.co/lHYXEOHY6C',
     RE_URL.sub(' urlweb ', '@bbcmtd Wholesale Markets ablaze http://t.co/lHYXEOHY6C')),
    ('Mención', '@bbcmtd Wholesale Markets ablaze',
     RE_MENCION.sub(' ', '@bbcmtd Wholesale Markets ablaze')),
    ('Hashtag: se quita el símbolo, se conserva la palabra',
     'Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all',
     RE_HASHTAG.sub(r' \1 ', 'Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all')),
    ('Emoticón, se borra en la limpieza de clasificación',
     'Ablaze for you Lord :D',
     _quitar_emoticones('Ablaze for you Lord :D')),
]

tabla_pasos = pd.DataFrame(ejemplos_pasos, columns=['paso', 'antes', 'despues'])
guardar_tabla(tabla_pasos, 'preproc_ejemplos_pasos.csv')
tabla_pasos

tabla guardada: preproc_ejemplos_pasos.csv


,paso,antes,despues
0,Mojibake y entidades HTML,Barbados #Bridgetown JAMAICA  Two cars set ablaze: SANTA CRUZ,Barbados #Bridgetown JAMAICA Two cars set ablaze: SANTA CRUZ
1,Entidad HTML,Rene Ablaze &amp; Jacinta - Secret 2k13 (Fallen Skies Edit),Rene Ablaze & Jacinta - Secret 2k13 (Fallen Skies Edit)
2,URL a token uniforme,@bbcmtd Wholesale Markets ablaze http://t.co/lHYXEOHY6C,@bbcmtd Wholesale Markets ablaze urlweb
3,Mención,@bbcmtd Wholesale Markets ablaze,Wholesale Markets ablaze
4,"Hashtag: se quita el símbolo, se conserva la palabra",Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all,Our Deeds are the Reason of this earthquake May ALLAH Forgive us all
5,"Emoticón, se borra en la limpieza de clasificación",Ablaze for you Lord :D,Ablaze for you Lord


**Interpretación.** El mojibake y las entidades HTML se limpian juntos porque ambos son
corrupción de codificación, no contenido. Las URLs no se borran: se reemplazan por el token
`urlweb`, porque el análisis exploratorio anterior mostró que la sola presencia de una URL
es el patrón más discriminativo de todo el corpus. Lo que sí es ruido es el texto exacto de
la URL, que además es distinto en cada tweet y no generaliza. Las menciones se borran por
completo: identifican a una cuenta, no aportan al tema. Los hashtags se tratan distinto de
las menciones a propósito: el símbolo `#` no aporta nada, pero la palabra que envuelve sí,
así que se conserva sin el símbolo.

In [4]:
ejemplo_neg = "@ablaze what time does your talk go until? I don't know if I can make it due to work."
print('crudo:      ', ejemplo_neg)
print('tokens crudos con nltk.word_tokenize:')
import nltk
print(' ', nltk.word_tokenize(ejemplo_neg.lower()))
print()
print('tokenizar de este módulo, mismos tokens, sin puntuación suelta:')
print(' ', tokenizar(ejemplo_neg.lower()))
print()
print('limpio, clasificación:', limpiar_clasificacion(ejemplo_neg))

crudo:       @ablaze what time does your talk go until? I don't know if I can make it due to work.
tokens crudos con nltk.word_tokenize:
  ['@', 'ablaze', 'what', 'time', 'does', 'your', 'talk', 'go', 'until', '?', 'i', 'do', "n't", 'know', 'if', 'i', 'can', 'make', 'it', 'due', 'to', 'work', '.']

tokenizar de este módulo, mismos tokens, sin puntuación suelta:
  ['ablaze', 'what', 'time', 'does', 'your', 'talk', 'go', 'until', 'i', 'do', "n't", 'know', 'if', 'i', 'can', 'make', 'it', 'due', 'to', 'work']

limpio, clasificación: time talk go n't know make due work


**Interpretación: por qué la tokenización usa el tokenizador de Treebank de NLTK y no una
separación por espacios.** La frase contiene `don't`, y separarla ingenuamente por espacios
dejaría el token `don't` pegado, con el apóstrofe adentro. El tokenizador de Treebank separa
las contracciones en dos piezas: `do` y `n't`. Esa segunda pieza, `n't`, es exactamente la
forma en que el conjunto `NEGACIONES` de `preprocesamiento.py` espera encontrar la partícula
de negación:

```
NEGACIONES = {"no", "not", "never", "nothing", "neither", "nor", "n't", "without", "none"}
```

`STOPWORDS_INGLES` se construye restándole ese conjunto a la lista estándar de NLTK en
inglés. Tres de esas negaciones, `no`, `not` y `nor`, sí forman parte de la lista estándar de
NLTK, así que sin esta resta se perderían durante la limpieza. La tarea de hoy es de tema y
no de opinión, y en principio no depende de las negaciones, pero se deja la misma regla en
las dos limpiezas por consistencia con la limpieza de sentimiento de la entrega final, donde
quitar una negación sí invertiría la polaridad de la frase.

## Stopwords de dominio

Además de la lista estándar de NLTK en inglés, conviene revisar si hay palabras que
saturan esta colección en particular sin distinguir clase. El criterio que se usa aquí es
explícito y reproducible: sobre el corpus ya sin stopwords estándar, se cuenta en cuántos
tweets aparece cada palabra al menos una vez, por clase, y se calcula el logaritmo de la
razón de probabilidades entre clases con suavizado de Laplace:

```
logratio(palabra) = log( P(palabra | desastre) / P(palabra | no_desastre) )
```

Una palabra es candidata a stopword de dominio si aparece en más del 1% de los tweets y su
`|logratio|` es menor a 0.25: frecuente y repartida casi por igual entre las dos clases.

In [5]:
def prelimpio_para_stopwords(texto):
    t = texto.lower()
    t = RE_MOJIBAKE.sub(' ', t)
    t = RE_URL.sub(' ', t)
    t = RE_MENCION.sub(' ', t)
    t = RE_ENTIDAD_HTML.sub(' ', t)
    t = RE_HASHTAG.sub(r' \1 ', t)
    return [w for w in tokenizar(t) if w not in STOPWORDS_INGLES and len(w) > 1]

frec_doc = {0: collections.Counter(), 1: collections.Counter()}
n_por_clase = df.target.value_counts().to_dict()
for texto, tgt in zip(df.text, df.target):
    frec_doc[tgt].update(set(prelimpio_para_stopwords(texto)))

total_docs = len(df)
vocab = set(frec_doc[0]) | set(frec_doc[1])
filas = []
for w in vocab:
    d0, d1 = frec_doc[0][w], frec_doc[1][w]
    docfreq = (d0 + d1) / total_docs
    p0 = (d0 + 1) / (n_por_clase[0] + 2)
    p1 = (d1 + 1) / (n_por_clase[1] + 2)
    logratio = math.log(p1 / p0)
    filas.append({'palabra': w, 'doc_frecuencia': docfreq, 'logratio': logratio,
                   'docs_no_desastre': d0, 'docs_desastre': d1})

tabla_stop = pd.DataFrame(filas)
candidatas = tabla_stop[(tabla_stop.doc_frecuencia > 0.01) & (tabla_stop.logratio.abs() < 0.25)]
candidatas = candidatas.sort_values('doc_frecuencia', ascending=False).reset_index(drop=True)
guardar_tabla(candidatas.round(4), 'preproc_stopwords_dominio.csv')
candidatas

tabla guardada: preproc_stopwords_dominio.csv


,palabra,doc_frecuencia,logratio,docs_no_desastre,docs_desastre
0,no,0.032999,-0.001044,142,105
1,video,0.020307,-0.042565,89,63
2,still,0.016700,0.061033,70,55
3,us,0.015631,-0.093117,70,47
4,man,0.014696,0.047047,62,48
5,first,0.014295,0.095837,59,48
6,world,0.013894,-0.083573,62,42
7,rt,0.013627,-0.011793,59,43
8,say,0.011356,0.044127,48,37
9,could,0.011356,0.137093,46,39


**Interpretación.** El criterio devuelve `no` en primer lugar, pero `no` es una negación y
nunca se trata como stopword, sin importar cuán frecuente y pareja sea su distribución entre
clases: eliminarla invertiría la lectura de cualquier frase que la contenga. También aparece
`death` con logratio 0.220, apenas debajo del umbral, pero es una palabra con contenido
temático real y forma parte del vocabulario típico de un desastre, así que se excluye de la
lista final por criterio de dominio y no solo por el número.

Las palabras que sí se agregan a `STOPWORDS_DOMINIO` en `preprocesamiento.py` son:

```
{'video', 'still', 'us', 'man', 'rt', 'first', 'world', 'say', 'could'}
```

Vale la pena notar que este criterio, más estricto que una simple inspección de frecuencia
compartida, descarta candidatas que a primera vista parecen ruido de dominio. `people` y
`burning` tienen logratio 0.40 y 0.42 respectivamente: son compartidas por las dos clases,
pero no en la misma proporción, así que sí discriminan y no deben eliminarse. La lección es
la misma que la del cuaderno anterior: frecuencia e informatividad son cosas distintas, y
solo la segunda debería decidir qué es una stopword de dominio.

## Stemming y lematización

El módulo implementa las dos técnicas de reducción morfológica: `SnowballStemmer('english')`
para el stemming y `WordNetLemmatizer` para la lematización. Se comparan sobre 10 palabras
reales del corpus, elegidas porque su forma cambia de manera distinta entre las dos
técnicas.

In [6]:
palabras_muestra = [
    'earthquake', 'fires', 'running', 'bodies', 'crashed',
    'evacuated', 'flooding', 'burning', 'killed', 'disasters',
]

tabla_reduccion = pd.DataFrame({
    'original': palabras_muestra,
    'stem': [STEMMER.stem(w) for w in palabras_muestra],
    'lema': [LEMATIZADOR.lemmatize(w) for w in palabras_muestra],
    'lema_verbo': [LEMATIZADOR.lemmatize(w, pos='v') for w in palabras_muestra],
})
guardar_tabla(tabla_reduccion, 'preproc_stem_vs_lema.csv')
tabla_reduccion

tabla guardada: preproc_stem_vs_lema.csv


,original,stem,lema,lema_verbo
0,earthquake,earthquak,earthquake,earthquake
1,fires,fire,fire,fire
2,running,run,running,run
3,bodies,bodi,body,body
4,crashed,crash,crashed,crash
5,evacuated,evacu,evacuated,evacuate
6,flooding,flood,flooding,flood
7,burning,burn,burning,burn
8,killed,kill,killed,kill
9,disasters,disast,disaster,disasters


**Interpretación.** El stemming es una operación puramente mecánica: corta sufijos según
reglas y no le importa si el resultado es una palabra real. Por eso `disasters` se convierte
en `disast` y `flooding` en `flood`, pero también corta de más en casos donde el resultado ya
no se reconoce como palabra del idioma. La lematización, en cambio, busca la forma canónica
en el diccionario de WordNet y por defecto asume que la palabra es un sustantivo: por eso
`running` y `crashed` no cambian en la columna `lema`, porque como sustantivos ya son su
propia forma base. Al indicarle explícitamente que la palabra es un verbo, en la columna
`lema_verbo`, `running` se convierte en `run` y `crashed` en `crash`.

Esta comparación es la razón por la que `pipeline_texto` usa lematización por defecto, con
`reduccion='lema'`, para `text_clean`. Conserva palabras reales y es más interpretable al
leer el vocabulario resultante, aunque `text_stem` se deja disponible con el resultado del
stemming para poder comparar el efecto de ambas técnicas sobre el modelo en la entrega
final.

## Aplicación al conjunto completo

Se generan tres columnas de texto procesado: `text_clean` con lematización, `text_stem` con
stemming y `text_sent` con la limpieza conservadora de sentimiento. Las dos primeras se
generan con `quitar_numeros=False`. El análisis exploratorio anterior mostró que los dígitos
son señal discriminativa: 72.74% de los tweets de desastre los contienen, contra 49.90% del
resto. Por eso no se eliminan a ciegas, pese a que el parámetro por defecto de
`pipeline_texto` los quita.

In [7]:
df['text_clean'] = df['text'].apply(
    lambda t: ' '.join(pipeline_texto(t, modo='clasificacion', reduccion='lema', quitar_numeros=False)))
df['text_stem'] = df['text'].apply(
    lambda t: ' '.join(pipeline_texto(t, modo='clasificacion', reduccion='stem', quitar_numeros=False)))
df['text_sent'] = df['text'].apply(limpiar_sentimiento)

print('Listo. Columnas del DataFrame:', list(df.columns))

Listo. Columnas del DataFrame: ['id', 'keyword', 'location', 'text', 'target', 'n_palabras', 'n_caracteres', 'text_clean', 'text_stem', 'text_sent']


In [8]:
vacios = df[df.text_clean.str.len() == 0]
print(f'Tweets con text_clean vacío: {len(vacios)} de {len(df):,} ({len(vacios)/len(df)*100:.3f}%)')
vacios[['id', 'text', 'target']]

Tweets con text_clean vacío: 1 de 7,485 (0.013%)


,id,text,target
15,23,What's up man?,0


**Sobre los registros vacíos.** Un solo tweet, `"What's up man?"`, queda sin ningún token
después de la limpieza de clasificación: `what` y `up` son stopwords estándar del inglés y
`man` es una stopword de dominio de esta colección. Es un tweet de la clase `no desastre`,
con `target = 0`, y su vacío es coherente con lo que dice: no tiene ningún contenido temático
que reportar. La fila **no se elimina**: se conserva con `text_clean` vacío, porque
descartarla rompería la alineación con `target` y con las demás columnas, y porque un vector
de características todo en cero es una representación válida, aunque poco informativa, de un
tweet sin señal temática. Es un único caso sobre 7,485 filas y no afecta de forma medible
ningún resultado agregado.

In [9]:
pd.set_option('display.max_colwidth', 70)
ejemplos_finales = df.sample(10, random_state=config.SEED)[['text', 'text_clean', 'text_stem']]
ejemplos_finales

,text,text_clean,text_stem
2826,BLOG: Rain much needed as drought conditions worsen: Right now Cha...,blog rain much needed drought condition worsen right charlotte muc...,blog rain much need drought condit worsen right charlott much surr...
6956,Ancient Mayan Tablet with Hieroglyphics Honors Lowly King http://t...,ancient mayan tablet hieroglyphic honor lowly king urlweb,ancient mayan tablet hieroglyph honor lowli king urlweb
3014,There has not been 1 real tear out of #Shelli 's eyes this entire ...,not 1 real tear shelli eye entire episode bb17,not 1 real tear shelli eye entir episod bb17
6569,Ebay Snipe RT? http://t.co/SlQnph34Nt Lego Power Miners Set 8960 T...,ebay snipe urlweb lego power miner set 8960 thunder driller boxed ...,ebay snipe urlweb lego power miner set 8960 thunder driller box pl...
4606,#Flashflood causes #landslide in Gilgit #Pakistan Damage to 20 hom...,flashflood cause landslide gilgit pakistan damage 20 home farmland...,flashflood caus landslid gilgit pakistan damag 20 home farmland ro...
4355,@pmarca content is held hostage by network due to affiliation fees.,content held hostage network due affiliation fee,content held hostag network due affili fee
6813,The worst voice I can ever hear is the 'Nikki your in trouble' vo...,worst voice ever hear nikki trouble voice mom,worst voic ever hear nikki troubl voic mom
468,illegal alien released by Obama/DHS 4 times Charged With Rape &amp...,illegal alien released obama dhs 4 time charged rape murder santa ...,illeg alien releas obama dhs 4 time charg rape murder santa maria ...
3127,#EMERGENCY in Odai Bucharest Romania 600 Dogs Dying!They are so Hu...,emergency odai bucharest romania 600 dog dying hungry eat urlweb,emerg odai bucharest romania 600 dog die hungri eat urlweb
1295,RT @HuffPostComedy: We should build a wall that keeps Burning Man ...,build wall keep burning attendee coming home urlweb urlweb,build wall keep burn attende come home urlweb urlweb


**Interpretación.** La muestra confirma lo que se documentó paso a paso: las URLs se
vuelven `urlweb`, las menciones desaparecen, los hashtags conservan su palabra, los números
se conservan y las stopwords estándar más las de dominio quedan fuera. `text_stem` produce en
varios casos formas que ya no son palabras reales del inglés, como es de esperar de un
stemmer basado en reglas.

Con esto queda completo el preprocesamiento del ejercicio 3. El siguiente cuaderno,
`03_ngramas.ipynb`, construye sobre `text_clean` el análisis de unigramas, bigramas y
trigramas del ejercicio 4.